In [ ]:
!pip install pymongo dnspython pandas numpy --quiet

In [ ]:
# Default paths (relative to project root)
DEFAULT_SQLITE = Path("data/raw/RDS-2013-0009.5_SQLITE.sqlite")
DEFAULT_OUT    = Path("data/fires_raw.json")
LOG_PATH       = Path("logs/01_extract.log")

# CONUS bounding box (longitude, latitude)
CONUS_LON_MIN, CONUS_LON_MAX = -125.0, -66.5
CONUS_LAT_MIN, CONUS_LAT_MAX =   24.0,  49.5

# FPA FOD NWCG size-class thresholds (acres)
SIZE_CLASS_BREAKS = [
    (0,      0.25,  "A"),
    (0.25,   10.0,  "B"),
    (10.0,  100.0,  "C"),
    (100.0, 300.0,  "D"),
    (300.0, 1_000.0, "E"),
    (1_000.0, 5_000.0, "F"),
    (5_000.0, float("inf"), "G"),
]

# Broad cause mapping from FPA FOD STAT_CAUSE_CODE (1–13)
CAUSE_MAP = {
    1: "Lightning",
    2: "Human", 3: "Human", 4: "Human", 5: "Human",
    6: "Human", 7: "Human", 8: "Human", 9: "Human",
    10: "Human", 11: "Human", 12: "Human",
    13: "Unknown",
}

In [ ]:
def setup_logging(log_path: Path) -> logging.Logger:
    """Configure file + console logging and return the module logger."""
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(levelname)s  %(message)s",
        handlers=[
            logging.FileHandler(log_path, mode="w"),
            logging.StreamHandler(),
        ],
    )
    return logging.getLogger(__name__)

In [ ]:
def assign_size_class(acres: float) -> str:
    """
    Return the NWCG size class letter (A–G) for a given acreage.
    Falls back to 'G' for values above the largest defined break.
    """
    for lo, hi, cls in SIZE_CLASS_BREAKS:
        if lo <= acres < hi:
            return cls
    return "G"


def parse_date(date_str: str | None) -> str | None:
    """
    Convert a FPA FOD date string (YYYY-MM-DD or YYYY/MM/DD) to ISO-8601.
    Returns None if the input is null or unparseable.
    """
    if not date_str:
        return None
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(date_str.strip(), fmt).strftime("%Y-%m-%dT00:00:00Z")
        except ValueError:
            continue
    return None  # could not parse


def containment_days(discovery: str | None, contain: str | None) -> float | None:
    """
    Compute the number of days between discovery and containment dates.
    Returns None when either date is missing or containment precedes discovery.
    """
    if not discovery or not contain:
        return None
    try:
        d = datetime.fromisoformat(discovery.rstrip("Z"))
        c = datetime.fromisoformat(contain.rstrip("Z"))
        delta = (c - d).days
        return float(delta) if delta >= 0 else None
    except ValueError:
        return None


def is_in_conus(lon: float, lat: float) -> bool:
    """Return True if the coordinate falls inside the CONUS bounding box."""
    return (
        CONUS_LON_MIN <= lon <= CONUS_LON_MAX
        and CONUS_LAT_MIN <= lat <= CONUS_LAT_MAX
    )

In [ ]:
def extract_fires(sqlite_path: Path, logger: logging.Logger) -> list[dict]:
    """
    Query the FPA FOD SQLite database, validate each row, and return a list
    of cleaned fire documents ready for JSON serialisation.

    Each returned document matches the wildfire soft schema defined in the
    project metadata (see README Metadata section).

    Args:
        sqlite_path: Path to the FPA FOD .sqlite file.
        logger:      Module logger for progress and warning messages.

    Returns:
        List of dicts, one per valid CONUS fire record.
    """
    if not sqlite_path.exists():
        raise FileNotFoundError(
            f"SQLite file not found: {sqlite_path}\n"
            "Download from https://www.fs.usda.gov/rds/archive/catalog/RDS-2013-0009.5"
        )

    logger.info("Opening SQLite database: %s", sqlite_path)

    try:
        con = sqlite3.connect(sqlite_path)
        con.row_factory = sqlite3.Row   # lets us access columns by name
        cur = con.cursor()
    except sqlite3.Error as exc:
        logger.error("Failed to open SQLite database: %s", exc)
        raise

    # Minimal column set we need from the Fires table
    query = """
        SELECT
            FPA_ID,
            FIRE_NAME,
            FIRE_YEAR,
            DISCOVERY_DATE,
            CONT_DATE,
            FIRE_SIZE,
            STAT_CAUSE_CODE,
            LATITUDE,
            LONGITUDE,
            STATE,
            NWCG_REPORTING_UNIT_ID
        FROM Fires
        ORDER BY DISCOVERY_DATE
    """

    try:
        cur.execute(query)
    except sqlite3.OperationalError as exc:
        logger.error("SQL query failed: %s", exc)
        con.close()
        raise

    documents = []
    skipped   = 0
    total_rows = 0

    for row in cur:
        total_rows += 1

        # ── Validate required fields ──────────────────────────────────────
        fpa_id = row["FPA_ID"]
        lat    = row["LATITUDE"]
        lon    = row["LONGITUDE"]
        size   = row["FIRE_SIZE"]

        # Skip records with missing geometry or size
        if lat is None or lon is None or size is None:
            logger.warning("Skipped %s — missing lat/lon/size", fpa_id)
            skipped += 1
            continue

        # Skip records outside CONUS
        if not is_in_conus(float(lon), float(lat)):
            skipped += 1
            continue

        # Skip negative or zero acreage (data entry error)
        if float(size) <= 0:
            logger.warning("Skipped %s — non-positive fire size: %s", fpa_id, size)
            skipped += 1
            continue

        # ── Build the canonical document ──────────────────────────────────
        acres             = float(size)
        discovery_iso     = parse_date(row["DISCOVERY_DATE"])
        containment_iso   = parse_date(row["CONT_DATE"])
        days_to_contain   = containment_days(discovery_iso, containment_iso)
        size_log          = round(math.log10(acres + 1), 6)
        cause_code        = row["STAT_CAUSE_CODE"]
        ignition_cause    = CAUSE_MAP.get(int(cause_code) if cause_code else 13, "Unknown")
        fire_name         = (row["FIRE_NAME"] or "UNNAMED").strip().upper()

        doc = {
            "_id":              fpa_id,                          # unique FPA FOD identifier
            "fire_name":        fire_name,                       # official incident name, UPPER CASE
            "fire_year":        int(row["FIRE_YEAR"]) if row["FIRE_YEAR"] else None,
            "discovery_date":   discovery_iso,                   # ISO-8601 UTC string
            "containment_date": containment_iso,                 # ISO-8601 UTC string (may be None)
            "final_size_acres": acres,                           # acres at containment
            "containment_days": days_to_contain,                 # derived duration (may be None)
            "size_log":         size_log,                        # log10(acres+1) for modelling
            "size_class":       assign_size_class(acres),        # NWCG A–G classification
            "ignition_cause":   ignition_cause,                  # broad ICS cause category
            "location": {
                "type":        "Point",                          # GeoJSON type — always Point
                "coordinates": [float(lon), float(lat)],         # WGS-84: [longitude, latitude]
            },
            "state":       (row["STATE"] or "").strip().upper(), # 2-letter USPS abbreviation
            "gacc_region": (row["NWCG_REPORTING_UNIT_ID"] or "").strip() or None,
            # weather sub-document populated in step 2 (02_fetch_era5_weather.py)
            "weather":     None,
            # fuel model and incident type populated manually or from ICS-209 (if available)
            "fuel_model":    None,
            "incident_type": None,
        }

        documents.append(doc)

        # Progress heartbeat every 100 000 records
        if len(documents) % 100_000 == 0:
            logger.info("  ... extracted %d valid records so far", len(documents))

    con.close()

    logger.info(
        "Extraction complete — total rows: %d | valid CONUS: %d | skipped: %d",
        total_rows, len(documents), skipped,
    )
    return documents

In [ ]:
def write_ndjson(documents: list[dict], out_path: Path, logger: logging.Logger) -> None:
    """
    Write a list of dicts to a newline-delimited JSON file (one JSON object
    per line). This format streams cleanly into mongoimport and pandas.

    Args:
        documents: List of fire document dicts.
        out_path:  Destination file path.
        logger:    Module logger.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with out_path.open("w", encoding="utf-8") as fh:
            for doc in documents:
                fh.write(json.dumps(doc, default=str) + "\n")
    except OSError as exc:
        logger.error("Failed to write output file %s: %s", out_path, exc)
        raise

    logger.info("Wrote %d documents to %s", len(documents), out_path)

In [ ]:
def parse_args() -> argparse.Namespace:
    """Parse command-line arguments."""
    parser = argparse.ArgumentParser(
        description="Extract FPA FOD wildfire records from SQLite and export to NDJSON."
    )
    parser.add_argument(
        "--sqlite", type=Path, default=DEFAULT_SQLITE,
        help=f"Path to FPA FOD .sqlite file (default: {DEFAULT_SQLITE})"
    )
    parser.add_argument(
        "--out", type=Path, default=DEFAULT_OUT,
        help=f"Output NDJSON path (default: {DEFAULT_OUT})"
    )
    return parser.parse_args()


def main() -> None:
    """Main entry point — orchestrates extraction and writing."""
    args   = parse_args()
    logger = setup_logging(LOG_PATH)
    logger.info("=== 01_extract_fpa_fod.py started ===")
    logger.info("Input  : %s", args.sqlite)
    logger.info("Output : %s", args.out)

    try:
        documents = extract_fires(args.sqlite, logger)
        write_ndjson(documents, args.out, logger)
    except Exception as exc:
        logger.error("Fatal error: %s", exc)
        raise

    logger.info("=== 01_extract_fpa_fod.py finished ===")


if __name__ == "__main__":
    main()